# 04 — dbt point-in-time runs with the alethe macro shim

This notebook builds a **real dbt project** (dbt-duckdb), drops in
[`dbt_macros/alethe_pit.sql`](../dbt_macros/alethe_pit.sql), and shows the
one-flag PIT workflow:

```bash
dbt run -s revenue --vars '{"alethe_as_of": "2024-03-01"}'
```

With the var **unset**, compilation is byte-identical to stock dbt.
With it **set**:
- `{{ source(...) }}` → the relation + engine-native time travel
- `{{ ref(...) }}` of a **snapshot** → a `dbt_valid_from`/`dbt_valid_to` validity subquery
- `{{ ref(...) }}` of a model → unchanged (models are derived; their sources are already bound)

**Extra requirements:** `pip install dbt-core dbt-duckdb`

In [1]:
import json, shutil, subprocess, sys
from pathlib import Path

PROJ = Path("dbt_pit_workspace").resolve()
if PROJ.exists():
    shutil.rmtree(PROJ)
for sub in ("models", "macros", "snapshots"):
    (PROJ / sub).mkdir(parents=True)

def dbt(*args):
    """Run a dbt command in the project dir; return combined output."""
    r = subprocess.run(
        [sys.executable, "-m", "dbt.cli.main", *args, "--profiles-dir", "."],
        cwd=PROJ, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:]); print(r.stderr[-500:])
        raise RuntimeError(f"dbt {' '.join(args)} failed")
    return r.stdout

print("workspace:", PROJ)

workspace: /Users/seamusaran/Documents/alethe/notebooks/dbt_pit_workspace


## 1. The project

Two sources (`raw.orders`, `raw.customers`), one snapshot over customers,
one staging model, one mart joining staging + snapshot:

```
source raw.orders ──► stg_orders ──┐
                                   ├──► revenue
source raw.customers ─► cust_snap ─┘   (snapshot, SCD2)
```

In [2]:
(PROJ / "dbt_project.yml").write_text("""\
name: pit_demo
version: '1.0'
profile: pit_demo
model-paths: ["models"]
macro-paths: ["macros"]
snapshot-paths: ["snapshots"]
""")

(PROJ / "profiles.yml").write_text("""\
pit_demo:
  target: dev
  outputs:
    dev:
      type: duckdb
      path: pit_demo.duckdb
""")

(PROJ / "models" / "schema.yml").write_text("""\
version: 2
sources:
  - name: raw
    schema: raw
    tables:
      - name: orders
        meta:
          alethe:
            chain: delta://orders     # ties this source to its watermark chain
      - name: customers
""")

(PROJ / "models" / "stg_orders.sql").write_text(
    "select * from {{ source('raw', 'orders') }}\n")

(PROJ / "models" / "revenue.sql").write_text("""\
select o.customer_id, c.segment, sum(o.amount) as revenue
from {{ ref('stg_orders') }} o
join {{ ref('cust_snap') }} c on o.customer_id = c.customer_id
group by 1, 2
""")

(PROJ / "snapshots" / "cust_snap.sql").write_text("""\
{% snapshot cust_snap %}
{{ config(unique_key='customer_id', strategy='check', check_cols='all',
          target_schema='snapshots') }}
select * from {{ source('raw', 'customers') }}
{% endsnapshot %}
""")

# Drop in the alethe macro shim — this is the entire integration
shutil.copy("../dbt_macros/alethe_pit.sql", PROJ / "macros" / "alethe_pit.sql")
print("project files written; alethe_pit.sql installed in macros/")

project files written; alethe_pit.sql installed in macros/


In [3]:
# Seed the raw schema with real data (in a real stack these are your lake tables)
import duckdb
con = duckdb.connect(str(PROJ / "pit_demo.duckdb"))
con.execute("create schema if not exists raw")
con.execute("""
    create or replace table raw.orders as
    select 'O' || i as order_id, 'C' || (i % 3) as customer_id,
           100.0 + i as amount
    from range(9) t(i)
""")
con.execute("""
    create or replace table raw.customers as
    select 'C' || i as customer_id,
           case when i = 0 then 'enterprise' else 'smb' end as segment
    from range(3) t(i)
""")
con.close()
print("raw.orders + raw.customers seeded")

raw.orders + raw.customers seeded


## 2. Baseline: snapshot + run, no PIT var

Everything compiles and runs exactly as stock dbt — the shim is inert
until `alethe_as_of` is set.

In [4]:
out = dbt("snapshot")
print([l for l in out.splitlines() if "OK" in l or "snapshot" in l.lower()][-2:])
out = dbt("run")
print([l for l in out.splitlines() if "OK" in l][-2:])

baseline = (PROJ / "target" / "compiled" / "pit_demo" / "models" / "revenue.sql").read_text()
print("\n--- compiled revenue.sql (baseline) ---")
print(baseline)

['\x1b15:40:56  1 of 1 OK snapshotted snapshots.cust_snap ...................................... [\x1bOK\x1b in 0.04s]', '\x1b15:40:56  Finished running 1 snapshot in 0 hours 0 minutes and 0.09 seconds (0.09s).']


['\x1b15:40:58  1 of 2 OK created sql view model main.stg_orders ............................... [\x1bOK\x1b in 0.03s]', '\x1b15:40:58  2 of 2 OK created sql view model main.revenue .................................. [\x1bOK\x1b in 0.06s]']

--- compiled revenue.sql (baseline) ---
select o.customer_id, c.segment, sum(o.amount) as revenue
from "pit_demo"."main"."stg_orders" o
join "pit_demo"."snapshots"."cust_snap" c on o.customer_id = c.customer_id
group by 1, 2


## 3. The PIT run: one `--vars` flag

`alethe_as_of_style: spark` tells the shim which time-travel dialect to
render (duckdb has none of its own — on Databricks or Trino you'd omit it,
the shim detects the adapter).

In [5]:
dbt("compile", "--vars",
    '{"alethe_as_of": "2024-03-01", "alethe_as_of_style": "spark"}')

print("--- compiled stg_orders.sql (as of 2024-03-01) ---")
print((PROJ / "target" / "compiled" / "pit_demo" / "models" / "stg_orders.sql").read_text())
print("--- compiled revenue.sql (as of 2024-03-01) ---")
print((PROJ / "target" / "compiled" / "pit_demo" / "models" / "revenue.sql").read_text())

--- compiled stg_orders.sql (as of 2024-03-01) ---
select * from "pit_demo"."raw"."orders" TIMESTAMP AS OF '2024-03-01'
--- compiled revenue.sql (as of 2024-03-01) ---
select o.customer_id, c.segment, sum(o.amount) as revenue
from "pit_demo"."main"."stg_orders" o
join (select * from "pit_demo"."snapshots"."cust_snap" where dbt_valid_from <= '2024-03-01' and (dbt_valid_to > '2024-03-01' or dbt_valid_to is null)) c on o.customer_id = c.customer_id
group by 1, 2


Note the two different temporal mechanisms, chosen automatically:

- **`raw.orders`** (a physical source) got `TIMESTAMP AS OF` — storage-level
  time travel against the Delta/Iceberg table.
- **`cust_snap`** (a dbt snapshot) got a **validity-window subquery** over
  `dbt_valid_from`/`dbt_valid_to`. Snapshots keep history in *rows*;
  time-travelling the snapshot table would answer "what did the snapshot
  look like at t", not "what was the customer's segment at t".
- **`stg_orders`** (a model ref) is untouched — it's derived, and its own
  source reference is already bound.

## 4. Gate it in CI

The macro binds the query but can't know whether 2024-03-01 is honestly
answerable — that's the PIT report's job. In CI, before the dbt run:

```bash
alethe check \
  --dbt-manifest target/manifest.json \
  --model revenue \
  --as-of 2024-03-01 \
  --watermarks s3://audit-bucket/watermarks.jsonl \
  && dbt run -s revenue --vars '{"alethe_as_of": "2024-03-01"}'
```

Exit `0` CERTAIN · `1` BOUNDED (`--allow-bounded` to accept) · `2` UNACHIEVABLE.
The `meta.alethe.chain` annotation in `schema.yml` (section 1) is how the
checker maps each dbt source to its recorded watermark chain.

Notebook **05** shows the same gate driving Airflow backfills.